# Consensus Explanation Walkthrough

This notebook demonstrates how XAI-Framework combines multiple
explanation methods and reports how much they agree.

We will cover:

1. A local explanation using a Random Forest.
2. A case where explanation methods disagree.
3. Global explanations with and without ground-truth labels.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from xai_framework import explain, explain_global


# Load a small scikit-learn dataset
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

# Split into training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Train a Random Forest classifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)

model.fit(X_train, y_train)

print("Model trained successfully.")
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

In [ ]:
# Explain one test instance using multiple explanation methods
instance = X_test.iloc[0]

result = explain(
    model,
    X_train,
    instance=instance,
    random_state=42,
)

result

In [ ]:
# Display the local explanation plot
result.plot()

In [ ]:
# Show the agreement between the explanation methods
result.agreement_matrix

In [ ]:
# Plot the consensus explanation
result.plot()

In [ ]:
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier

# Create a noisy dataset
X_noisy, y_noisy = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=3,
    n_redundant=2,
    flip_y=0.25,
    random_state=42,
)

# Train an MLP classifier
mlp = MLPClassifier(
    hidden_layer_sizes=(20,),
    max_iter=1000,
    random_state=42,
)

mlp.fit(X_noisy, y_noisy)

print("MLP model trained successfully.")

In [ ]:
# Explain one instance using the MLP model
noisy_instance = X_noisy[1]

mlp_result = explain(
    mlp,
    X_noisy,
    instance=noisy_instance,
    random_state=42,
)

mlp_result

In [ ]:
# Show the explanation plot
mlp_result.plot()


In [ ]:
# Show the agreement matrix
mlp_result.agreement_matrix

In [ ]:
# Increase the number of samples used by the surrogate explainer
mlp_result_more_samples = explain(
    mlp,
    X_noisy,
    instance=noisy_instance,
    random_state=42,
    explainer_kwargs={
        "surrogate": {
            "n_samples": 100000
        }
    },
)

mlp_result_more_samples

### Effect of increasing surrogate samples

For this noisy instance, the default surrogate explanation shows substantial disagreement
between the coalition and surrogate explanations, with an agreement of about 0.24.

After increasing the surrogate sample size to 100,000, the agreement changes to about 0.18.
The agreement does not necessarily improve when the surrogate sample size is increased,
but the change shows that surrogate sampling settings can affect the explanation.

For this instance, f8 remains the strongest positive feature, while f7 has a negative
contribution. Both explanations target class 0.

In [ ]:
# Global explanation without ground-truth labels
global_without_y = explain_global(
    mlp,
    X_noisy,
)

global_without_y

In [ ]:
# Global explanation with ground-truth labels
global_with_y = explain_global(
    mlp,
    X_noisy,
    y=y_noisy,
)

global_with_y